# Tutorial de uso do WebSensors Flow RAG

Este tutorial parte de uma API já em execução e de uma infraestrutura pronta para receber documentos.

O fluxo começa com novos documentos no MinIO Bronze. Em seguida são executadas indexação, busca BM25, busca semântica e busca híbrida com RRF.

Os exemplos também mostram filtros, atualização de metadados, atualização de conteúdo e uma comparação simples de ranking.

## 1. Bibliotecas

As células usam HTTP para conversar com a API, MinIO para enviar os documentos de exemplo, pandas para organizar os resultados e matplotlib para os gráficos.

In [ ]:
import io
import json
import math
import time
from pathlib import Path
from urllib.parse import urlparse

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display
from minio import Minio

pd.set_option("display.max_colwidth", 120)

## 2. Conexões

O endereço da API e do MinIO pode ser ajustado nesta célula.

As credenciais e os nomes dos buckets são lidos do mesmo YAML usado pelo serviço.

In [ ]:
API_URL = "http://127.0.0.1:8000"
MINIO_URL = "http://127.0.0.1:9000"
PREFIXO = "tutoriais/rag"

candidatos = [
    Path("config/rag.local.yaml"),
    Path("../config/rag.local.yaml"),
    Path("config/rag.yaml"),
    Path("../config/rag.yaml"),
]
ARQUIVO_YAML = next((p.resolve() for p in candidatos if p.exists()), None)
if ARQUIVO_YAML is None:
    raise FileNotFoundError("Arquivo YAML não encontrado")

CONFIG = yaml.safe_load(ARQUIVO_YAML.read_text(encoding="utf-8"))
api_auth = CONFIG.get("api", {}).get("auth", {})
api_token = api_auth.get("token") if api_auth.get("enabled") else None
headers = {"Authorization": f"Bearer {api_token}"} if api_token else {}
api = httpx.Client(base_url=API_URL, headers=headers, timeout=120.0)

minio_cfg = CONFIG["services"]["minio"]
parsed = urlparse(MINIO_URL)
minio = Minio(
    parsed.netloc,
    access_key=minio_cfg["access_key"],
    secret_key=minio_cfg["secret_key"],
    secure=parsed.scheme == "https",
    region=minio_cfg.get("region"),
)
BRONZE = minio_cfg["bronze_bucket"]
SILVER = minio_cfg["silver_bucket"]

print("API", API_URL)
print("MinIO", MINIO_URL)
print("Bronze", BRONZE)
print("Silver", SILVER)

## 3. Funções para chamar a API

A indexação é assíncrona. A primeira requisição devolve um token.

A função de acompanhamento consulta o token até o término e mostra step, percentual, tempo decorrido e ETA.

In [ ]:
def chamar_api(metodo, rota, dados=None, parametros=None):
    resposta = api.request(metodo, rota, json=dados, params=parametros)
    if resposta.status_code >= 400:
        print(resposta.text[:3000])
    resposta.raise_for_status()
    return resposta.json()


def iniciar_indexacao(chave, metadata=None, force=False):
    corpo = {"key": chave, "force": force}
    if metadata is not None:
        corpo["metadata"] = metadata
    return chamar_api("POST", "/documents/index", corpo)


def acompanhar_indexacao(token, intervalo=1.5, limite_segundos=1200):
    inicio = time.monotonic()
    ultimo = None
    while True:
        estado = chamar_api("GET", f"/indexing/{token}")
        progresso = estado.get("progress") or {}
        assinatura = (
            estado.get("status"),
            progresso.get("current_step"),
            progresso.get("percent"),
            progresso.get("eta_seconds"),
        )
        if assinatura != ultimo:
            eta = progresso.get("eta_seconds")
            eta_texto = f"{eta:.1f}s" if isinstance(eta, (int, float)) else "calculando"
            print(
                "estado", estado.get("status"),
                "step", progresso.get("current_step"),
                "progresso", f"{progresso.get('percent', 0):.1f}%",
                "ETA", eta_texto,
            )
            loop = progresso.get("loop")
            if loop:
                loop_eta = loop.get("eta_seconds")
                loop_eta_texto = f"{loop_eta:.1f}s" if isinstance(loop_eta, (int, float)) else "calculando"
                print(
                    "  operação", loop.get("description"),
                    f"{loop.get('current', 0)}/{loop.get('total', 0)}",
                    "ETA", loop_eta_texto,
                )
            ultimo = assinatura
        if estado.get("terminal"):
            if estado.get("status") != "success":
                raise RuntimeError(json.dumps(estado.get("error"), ensure_ascii=False, indent=2))
            return estado
        if time.monotonic() - inicio > limite_segundos:
            raise TimeoutError("Tempo máximo de acompanhamento excedido")
        time.sleep(intervalo)


def indexar_e_aguardar(chave, metadata=None, force=False):
    inicio = iniciar_indexacao(chave, metadata=metadata, force=force)
    print("documento", inicio.get("document_id"))
    print("token", inicio.get("token"))
    final = acompanhar_indexacao(inicio["token"])
    resultado = final.get("result") or {}
    print("ação", resultado.get("action"))
    return final

## 4. Coleção de exemplo

A coleção contém documentos relacionados e documentos de controle.

Isso permite observar diferenças entre busca lexical, busca semântica e busca híbrida.

In [ ]:
DOCUMENTOS = {
    "aquecimento": {
        "metadata": {"colecao": "tutorial_rag", "tema": "termica", "ano": 2026},
        "texto": """# Compensação térmica durante o aquecimento

## Regime transiente

Nos primeiros minutos de operação a máquina apresenta aquecimento rápido e comportamento térmico transiente. A relação entre temperaturas e deslocamentos pode ser não linear. Modelos simples apresentam maior erro antes da estabilização.

## Avaliação

O experimento compara janelas de 30, 60, 90, 120 e 180 minutos. O objetivo é identificar quando o comportamento passa a ser aproximadamente linear e quando a compensação térmica se torna mais estável.
""",
    },
    "estabilizacao": {
        "metadata": {"colecao": "tutorial_rag", "tema": "termica", "ano": 2026},
        "texto": """# Regime térmico estável

## Estabilização

Depois do período inicial de aquecimento as temperaturas variam mais lentamente. Nesse regime a relação entre temperatura e deslocamento tende a ser aproximadamente linear.

## Modelagem

Uma regressão linear pode ser suficiente para a fase estável. A simplicidade do modelo reduz custo computacional e facilita manutenção.
""",
    },
    "sensores": {
        "metadata": {"colecao": "tutorial_rag", "tema": "sensores", "ano": 2026},
        "texto": """# Sensores e condições de operação

## Rotação

Os experimentos registram RPM, temperaturas internas e externas e deslocamentos nos eixos. Diferentes rotações podem alterar a velocidade de aquecimento e o padrão térmico observado.

## Qualidade dos dados

Sincronização temporal e identificação de leituras ausentes são necessárias antes do treinamento dos modelos.
""",
    },
    "fraude": {
        "metadata": {"colecao": "tutorial_rag", "tema": "fraude", "ano": 2026},
        "texto": """# Detecção de fraude financeira

Transações podem ser representadas por grafos de contas, dispositivos e beneficiários. Padrões de conectividade ajudam a identificar grupos suspeitos e relações incomuns.
""",
    },
    "agricultura": {
        "metadata": {"colecao": "tutorial_rag", "tema": "agricultura", "ano": 2026},
        "texto": """# Monitoramento agrícola

Sensores de umidade do solo e dados meteorológicos podem apoiar decisões de irrigação. Séries temporais ajudam a estimar necessidade de água em diferentes áreas.
""",
    },
}

print("documentos", len(DOCUMENTOS))

## 5. Envio ao MinIO Bronze

Cada texto é gravado como Markdown no bucket Bronze.

A indexação será solicitada logo depois.

In [ ]:
CHAVES = {}
for nome, item in DOCUMENTOS.items():
    chave = f"{PREFIXO}/{nome}.md"
    dados = item["texto"].encode("utf-8")
    minio.put_object(
        BRONZE,
        chave,
        io.BytesIO(dados),
        length=len(dados),
        content_type="text/markdown",
    )
    CHAVES[nome] = chave
    print(nome, chave)

## 6. Indexação do primeiro documento

O primeiro exemplo mostra o acompanhamento completo do token.

Observe a mudança de step e o ETA calculado pela execução.

In [ ]:
primeiro = indexar_e_aguardar(
    CHAVES["aquecimento"],
    metadata=DOCUMENTOS["aquecimento"]["metadata"],
)

print(json.dumps(primeiro.get("result"), ensure_ascii=False, indent=2))

## 7. Indexação dos demais documentos

Os outros documentos são indexados pela mesma API.

In [ ]:
INDEXACOES = {"aquecimento": primeiro}
for nome in ["estabilizacao", "sensores", "fraude", "agricultura"]:
    print("Documento", nome)
    INDEXACOES[nome] = indexar_e_aguardar(
        CHAVES[nome],
        metadata=DOCUMENTOS[nome]["metadata"],
    )

## 8. Catálogo de documentos

A rota de documentos permite conferir o estado atual da coleção.

In [ ]:
catalogo = chamar_api("GET", "/documents", parametros={"limit": 100})
df_catalogo = pd.DataFrame(catalogo["documents"])
colunas = [c for c in ["document_id", "filename", "active_revision", "status", "updated_at"] if c in df_catalogo.columns]
display(df_catalogo[colunas])

## 9. Artefatos no MinIO Silver

Cada revisão completa grava Markdown, JSON e manifesto.

A célula abaixo localiza os artefatos do documento de aquecimento.

In [ ]:
doc_aquecimento = next(d for d in catalogo["documents"] if d.get("filename") == "aquecimento.md")
print(json.dumps(doc_aquecimento.get("silver"), ensure_ascii=False, indent=2))


def ler_s3(uri):
    parsed = urlparse(uri)
    resposta = minio.get_object(parsed.netloc, parsed.path.lstrip("/"))
    try:
        return resposta.read()
    finally:
        resposta.close()
        resposta.release_conn()

silver = doc_aquecimento.get("silver") or {}
if silver.get("markdown"):
    markdown = ler_s3(silver["markdown"]).decode("utf-8")
    print(markdown[:1200])

if silver.get("json"):
    estrutura = json.loads(ler_s3(silver["json"]).decode("utf-8"))
    print("JSON do Docling")
    print(json.dumps(estrutura, ensure_ascii=False, indent=2)[:1800])

if silver.get("manifest"):
    manifesto = json.loads(ler_s3(silver["manifest"]).decode("utf-8"))
    print("Manifesto da revisão")
    print(json.dumps(manifesto, ensure_ascii=False, indent=2)[:1800])


## 10. Funções de busca

As três modalidades usam o mesmo contrato de entrada.

Isso facilita comparar resultados com a mesma consulta, os mesmos filtros e o mesmo valor de top k.

In [ ]:
ROTAS = {
    "BM25": "/search/bm25",
    "Semântica": "/search/semantic",
    "Híbrida": "/search/hybrid",
}


def buscar(modo, consulta, top_k=5, filtros=None, debug=True):
    return chamar_api(
        "POST",
        ROTAS[modo],
        {
            "query": consulta,
            "top_k": top_k,
            "filters": filtros or {},
            "debug": debug,
        },
    )


def tabela_resultados(resposta, limite=180):
    linhas = []
    for hit in resposta["results"]:
        linhas.append({
            "posição": hit["rank"],
            "arquivo": hit.get("filename"),
            "seção": hit.get("section"),
            "página": hit.get("page_start"),
            "score": hit.get("score"),
            "trecho": " ".join(hit.get("content", "").split())[:limite],
        })
    print("modo", resposta["mode"])
    print("tempo em ms", resposta["took_ms"])
    if resposta.get("debug"):
        print("debug", json.dumps(resposta["debug"], ensure_ascii=False))
    display(pd.DataFrame(linhas))

## 11. Busca BM25

BM25 favorece termos presentes literalmente no texto.

A consulta usa RPM, uma expressão específica do documento de sensores.

In [ ]:
bm25 = buscar("BM25", "RPM sensores temperatura", top_k=5)
tabela_resultados(bm25)

## 12. Busca semântica

A busca semântica usa o significado da consulta.

A pergunta abaixo descreve a estabilização sem copiar o título do documento.

In [ ]:
semantica = buscar(
    "Semântica",
    "o que acontece com o modelo depois que a máquina termina de aquecer",
    top_k=5,
)
tabela_resultados(semantica)

## 13. Busca híbrida

A busca híbrida executa BM25 e kNN no Elasticsearch e combina os rankings com RRF.

O RRF trabalha com posições e não com a soma dos scores.

In [ ]:
CONSULTA = "compensação térmica durante os primeiros minutos de operação"
hibrida = buscar("Híbrida", CONSULTA, top_k=5)
tabela_resultados(hibrida)

## 14. Comparação dos três rankings

A mesma consulta é enviada para as três modalidades.

O quadro mostra a melhor posição de cada arquivo em cada resultado.

In [ ]:
RESPOSTAS = {modo: buscar(modo, CONSULTA, top_k=5) for modo in ROTAS}

linhas = []
for modo, resposta in RESPOSTAS.items():
    vistos = set()
    for hit in resposta["results"]:
        arquivo = hit.get("filename")
        if arquivo in vistos:
            continue
        vistos.add(arquivo)
        linhas.append({
            "método": modo,
            "arquivo": arquivo,
            "posição": hit["rank"],
            "tempo_ms": resposta["took_ms"],
        })

df_comparacao = pd.DataFrame(linhas)
display(df_comparacao.sort_values(["método", "posição"]))

## 15. Visualização do tempo observado

Uma única medição serve apenas para ilustrar a instrumentação.

Uma avaliação de desempenho precisa repetir as consultas e controlar cache e carga.

In [ ]:
df_tempo = pd.DataFrame([
    {"método": modo, "tempo_ms": resposta["took_ms"]}
    for modo, resposta in RESPOSTAS.items()
])
ax = df_tempo.plot.bar(x="método", y="tempo_ms", legend=False, figsize=(8, 4), rot=0)
ax.set_xlabel("Modalidade")
ax.set_ylabel("Tempo em ms")
ax.set_title("Tempo observado")
plt.tight_layout()
plt.show()

## 16. Busca com filtro

Os filtros restringem a coleção antes da recuperação.

O exemplo limita a consulta aos documentos térmicos.

In [ ]:
filtrada = buscar(
    "Híbrida",
    "comportamento durante aquecimento",
    top_k=5,
    filtros={"tema": "termica", "ano": 2026},
)
tabela_resultados(filtrada)

## 17. Atualização somente de metadados

A origem permanece igual.

O serviço pode atualizar os metadados ativos sem repetir a conversão documental e os embeddings.

In [ ]:
novos_metadados = dict(DOCUMENTOS["aquecimento"]["metadata"])
novos_metadados["revisado"] = True
novos_metadados["responsavel"] = "equipe_termica"

atualizacao_meta = indexar_e_aguardar(
    CHAVES["aquecimento"],
    metadata=novos_metadados,
)
print(json.dumps(atualizacao_meta.get("result"), ensure_ascii=False, indent=2))

## 18. Consulta do metadado atualizado

A busca híbrida pode filtrar pelo novo campo.

In [ ]:
resultado_revisado = buscar(
    "Híbrida",
    "erro no início da operação",
    top_k=5,
    filtros={"revisado": True},
)
tabela_resultados(resultado_revisado)

## 19. Atualização do conteúdo

Agora o arquivo de origem muda.

Isso produz uma nova revisão completa.

In [ ]:
texto_novo = DOCUMENTOS["aquecimento"]["texto"] + """

## Critério adicional

A transição para o regime estável será marcada quando o erro permanecer abaixo do limite experimental em janelas consecutivas.
"""

dados = texto_novo.encode("utf-8")
minio.put_object(
    BRONZE,
    CHAVES["aquecimento"],
    io.BytesIO(dados),
    length=len(dados),
    content_type="text/markdown",
)

atualizacao_conteudo = indexar_e_aguardar(
    CHAVES["aquecimento"],
    metadata=novos_metadados,
)
print(json.dumps(atualizacao_conteudo.get("result"), ensure_ascii=False, indent=2))

## 20. Repetição sem alterações

Repetir a mesma solicitação permite observar a detecção de ausência de mudanças.

In [ ]:
repeticao = indexar_e_aguardar(
    CHAVES["aquecimento"],
    metadata=novos_metadados,
)
print(json.dumps(repeticao.get("result"), ensure_ascii=False, indent=2))

## 21. Indexação opcional de PDF

A célula executa somente quando o arquivo local existe.

O PDF passa pelo mesmo endpoint de indexação e pelo mesmo fluxo do Docling.

In [ ]:
PDF_LOCAL = Path("exemplos/documento.pdf")
if PDF_LOCAL.exists():
    chave_pdf = f"{PREFIXO}/documento.pdf"
    conteudo = PDF_LOCAL.read_bytes()
    minio.put_object(BRONZE, chave_pdf, io.BytesIO(conteudo), len(conteudo), content_type="application/pdf")
    resultado_pdf = indexar_e_aguardar(chave_pdf, metadata={"colecao": "tutorial_rag", "tipo": "pdf"})
    print(json.dumps(resultado_pdf.get("result"), ensure_ascii=False, indent=2))
else:
    print("Arquivo opcional não encontrado", PDF_LOCAL)

## 22. Indexação opcional de imagem

Imagens também passam pelo Docling.

Quando a descrição visual está habilitada, o preset configurado no Docling participa da conversão.

In [ ]:
IMAGEM_LOCAL = Path("exemplos/figura.png")
if IMAGEM_LOCAL.exists():
    chave_imagem = f"{PREFIXO}/figura.png"
    conteudo = IMAGEM_LOCAL.read_bytes()
    minio.put_object(BRONZE, chave_imagem, io.BytesIO(conteudo), len(conteudo), content_type="image/png")
    resultado_imagem = indexar_e_aguardar(chave_imagem, metadata={"colecao": "tutorial_rag", "tipo": "imagem"})
    print(json.dumps(resultado_imagem.get("result"), ensure_ascii=False, indent=2))
else:
    print("Arquivo opcional não encontrado", IMAGEM_LOCAL)

## 23. Conjunto didático de relevância

As próximas células usam pequenos julgamentos de relevância para comparar os rankings.

Os valores são ilustrativos e servem para demonstrar MRR e nDCG.

In [ ]:
CASOS = [
    {
        "consulta": "erros nos primeiros minutos de aquecimento da máquina",
        "relevancia": {"aquecimento.md": 3, "sensores.md": 2, "estabilizacao.md": 1},
    },
    {
        "consulta": "influência de RPM nos sensores de temperatura",
        "relevancia": {"sensores.md": 3, "aquecimento.md": 2},
    },
    {
        "consulta": "grafos de contas para identificar fraude",
        "relevancia": {"fraude.md": 3},
    },
]
print("consultas", len(CASOS))

## 24. Métricas de ranking

MRR favorece o método que posiciona cedo o primeiro documento relevante.

nDCG considera diferentes graus de relevância e dá maior peso às primeiras posições.

In [ ]:
def arquivos_unicos(resposta):
    vistos = set()
    saida = []
    for hit in resposta["results"]:
        arquivo = hit.get("filename")
        if arquivo not in vistos:
            vistos.add(arquivo)
            saida.append(arquivo)
    return saida


def reciprocal_rank(resposta, relevancia):
    for posicao, arquivo in enumerate(arquivos_unicos(resposta), start=1):
        if relevancia.get(arquivo, 0) > 0:
            return 1.0 / posicao
    return 0.0


def ndcg(resposta, relevancia, k=5):
    ganhos = [relevancia.get(a, 0) for a in arquivos_unicos(resposta)[:k]]
    dcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(ganhos))
    ideal = sorted(relevancia.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

## 25. Comparação didática

Cada consulta é executada nos três métodos.

Os resultados mostram qualidade de ranking e tempo observado.

In [ ]:
avaliacoes = []
for numero, caso in enumerate(CASOS, start=1):
    for modo in ROTAS:
        resposta = buscar(
            modo,
            caso["consulta"],
            top_k=5,
            filtros={"colecao": "tutorial_rag"},
            debug=False,
        )
        avaliacoes.append({
            "consulta": numero,
            "método": modo,
            "rr": reciprocal_rank(resposta, caso["relevancia"]),
            "ndcg_5": ndcg(resposta, caso["relevancia"]),
            "tempo_ms": resposta["took_ms"],
        })

df_avaliacao = pd.DataFrame(avaliacoes)
display(df_avaliacao)

## 26. Resumo das métricas

A média do Reciprocal Rank produz MRR.

A média do nDCG resume a qualidade de ordenação no conjunto didático.

In [ ]:
resumo = (
    df_avaliacao.groupby("método", as_index=False)
    .agg(mrr=("rr", "mean"), ndcg_5=("ndcg_5", "mean"), tempo_medio_ms=("tempo_ms", "mean"))
)
display(resumo)

ax = resumo.plot.bar(x="método", y=["mrr", "ndcg_5"], figsize=(9, 4), rot=0)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Modalidade")
ax.set_ylabel("Valor médio")
ax.set_title("Comparação didática de ranking")
plt.tight_layout()
plt.show()

## 27. Inspeção do contrato OpenAPI

A célula lista as operações publicadas pela API.

In [ ]:
openapi = chamar_api("GET", "/openapi.json")
operacoes = []
for rota, metodos in openapi.get("paths", {}).items():
    for metodo, descricao in metodos.items():
        if isinstance(descricao, dict) and descricao.get("operationId"):
            operacoes.append({"método": metodo.upper(), "rota": rota, "operationId": descricao["operationId"]})
display(pd.DataFrame(operacoes).sort_values(["rota", "método"]))

## 28. Encerramento

O conjunto de células cobre o ciclo principal do serviço.

Para experimentos reais, use documentos do domínio, consultas anotadas e filtros equivalentes entre os métodos.

A busca híbrida é uma boa referência inicial quando lexicalidade e significado são importantes ao mesmo tempo.

In [ ]:
api.close()
print("Cliente HTTP encerrado")